In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

data = pd.read_csv("/content/data.csv")


In [ ]:
# print((data['requires_road_closure']==True).sum())

596


In [ ]:
data["start_datetime"] = pd.to_datetime(
    data["start_datetime"],
    errors="coerce"
)

data = data.dropna(subset=["start_datetime"]).copy()

latest_date = data["start_datetime"].max()

data["days_old"] = (
    latest_date - data["start_datetime"]
).dt.days

In [ ]:
HALF_LIFE_DAYS = 90

data["recency_weight"] = np.exp(
    -np.log(2) * data["days_old"] / HALF_LIFE_DAYS
)

display(
    data[
        ["event_cause", "start_datetime", "days_old", "recency_weight"]
    ]
)

,event_cause,start_datetime,days_old,recency_weight
0,vehicle_breakdown,2024-03-07 17:01:48.111000+00:00,32,0.781569
1,vehicle_breakdown,2024-01-30 04:07:24.173000+00:00,69,0.587774
2,others,2023-11-11 06:18:03.343000+00:00,149,0.317415
3,tree_fall,2024-03-07 17:56:55.061000+00:00,31,0.787611
4,vehicle_breakdown,2024-01-30 04:56:32.348000+00:00,69,0.587774
...,...,...,...,...
8168,vehicle_breakdown,2024-01-29 21:10:59.778000+00:00,69,0.587774
8169,vehicle_breakdown,2024-01-29 21:50:41.154000+00:00,69,0.587774
8170,vehicle_breakdown,2024-01-29 21:56:05.192000+00:00,69,0.587774
8171,pot_holes,2024-01-29 22:54:11.748000+00:00,69,0.587774


In [ ]:
event_stats = data.groupby("event_cause").agg(
    frequency=("event_cause", "size"),

    recency_score=(
        "recency_weight",
        "mean"
    ),

    road_closure_rate=(
        "requires_road_closure",
        lambda x: (x == True).mean()
    ),

    high_priority_rate=(
        "priority",
        lambda x: (x == "High").mean()
    )
).reset_index()

display(event_stats)

,event_cause,frequency,recency_score,road_closure_rate,high_priority_rate
0,Debris,12,0.540818,0.083333,0.666667
1,Fog / Low Visibility,2,0.510786,0.000000,0.500000
2,accident,365,0.558473,0.030137,0.460274
3,congestion,136,0.672674,0.044118,0.691176
4,construction,438,0.572184,0.223744,0.643836
5,debris,1,0.781569,1.000000,1.000000
6,others,636,0.585112,0.086478,0.595912
7,pot_holes,537,0.598774,0.024209,0.556797
8,procession,66,0.562732,0.227273,0.287879
9,protest,14,0.573878,0.428571,0.357143


In [ ]:
event_stats["severity_score"] = (
    0.6 * event_stats["road_closure_rate"]
    +
    0.4 * event_stats["high_priority_rate"]
)

display(event_stats)

,event_cause,frequency,recency_score,road_closure_rate,high_priority_rate,severity_score
0,Debris,12,0.540818,0.083333,0.666667,0.316667
1,Fog / Low Visibility,2,0.510786,0.000000,0.500000,0.200000
2,accident,365,0.558473,0.030137,0.460274,0.202192
3,congestion,136,0.672674,0.044118,0.691176,0.302941
4,construction,438,0.572184,0.223744,0.643836,0.391781
5,debris,1,0.781569,1.000000,1.000000,1.000000
6,others,636,0.585112,0.086478,0.595912,0.290252
7,pot_holes,537,0.598774,0.024209,0.556797,0.237244
8,procession,66,0.562732,0.227273,0.287879,0.251515
9,protest,14,0.573878,0.428571,0.357143,0.400000


In [ ]:
scaler = MinMaxScaler()

cols_to_normalize = [
    "frequency",
    "recency_score",
    "severity_score"
]

event_stats[
    [f"{c}_norm" for c in cols_to_normalize]
] = scaler.fit_transform(
    event_stats[cols_to_normalize]
)

display(event_stats)

,event_cause,frequency,recency_score,road_closure_rate,high_priority_rate,severity_score,frequency_norm,recency_score_norm,severity_score_norm
0,Debris,12,0.540818,0.083333,0.666667,0.316667,0.002252,0.176157,0.211538
1,Fog / Low Visibility,2,0.510786,0.000000,0.500000,0.200000,0.000205,0.087651,0.076923
2,accident,365,0.558473,0.030137,0.460274,0.202192,0.074514,0.228187,0.079452
3,congestion,136,0.672674,0.044118,0.691176,0.302941,0.027636,0.564734,0.195701
4,construction,438,0.572184,0.223744,0.643836,0.391781,0.089458,0.268592,0.298209
5,debris,1,0.781569,1.000000,1.000000,1.000000,0.000000,0.885646,1.000000
6,others,636,0.585112,0.086478,0.595912,0.290252,0.129990,0.306691,0.181060
7,pot_holes,537,0.598774,0.024209,0.556797,0.237244,0.109724,0.346953,0.119897
8,procession,66,0.562732,0.227273,0.287879,0.251515,0.013306,0.240737,0.136364
9,protest,14,0.573878,0.428571,0.357143,0.400000,0.002661,0.273583,0.307692


In [ ]:
event_stats["event_congestion_score"] = (
    0.50 * event_stats["recency_score_norm"]
    +
    0.30 * event_stats["severity_score_norm"]
    +
    0.20 * event_stats["frequency_norm"]
)

event_stats["event_congestion_score"] = (
    event_stats["event_congestion_score"] * 100
).round(2)

event_stats = event_stats.sort_values(
    "event_congestion_score",
    ascending=False
)

display(event_stats)

,event_cause,frequency,recency_score,road_closure_rate,high_priority_rate,severity_score,frequency_norm,recency_score_norm,severity_score_norm,event_congestion_score
5,debris,1,0.781569,1.000000,1.000000,1.000000,0.000000,0.885646,1.000000,74.28
13,tree_fall,284,0.820372,0.394366,0.327465,0.367606,0.057932,1.000000,0.270314,59.27
16,water_logging,458,0.798971,0.085153,0.591703,0.287773,0.093552,0.936931,0.178200,54.06
14,vehicle_breakdown,4886,0.601819,0.042571,0.662300,0.290463,1.000000,0.355927,0.181303,43.24
3,congestion,136,0.672674,0.044118,0.691176,0.302941,0.027636,0.564734,0.195701,34.66
4,construction,438,0.572184,0.223744,0.643836,0.391781,0.089458,0.268592,0.298209,24.16
11,road_conditions,170,0.601372,0.123529,0.547059,0.292941,0.034596,0.354609,0.184163,23.95
6,others,636,0.585112,0.086478,0.595912,0.290252,0.129990,0.306691,0.181060,23.37
7,pot_holes,537,0.598774,0.024209,0.556797,0.237244,0.109724,0.346953,0.119897,23.14
9,protest,14,0.573878,0.428571,0.357143,0.400000,0.002661,0.273583,0.307692,22.96


In [ ]:
event_stats.to_csv(
    "event_congestion_scores.csv",
    index=False
)

print(
    "Saved: event_congestion_scores.csv"
)

Saved: event_congestion_scores.csv
